# **DDL Script: Create Gold Views**

**Script Purpose**:
    
    This notebook creates views for the Gold layer in the data lakehouse.
    The Gold layer represents the final dimension and fact tables (Star Schema)

    Each view performs transformations and combines data from the Silver layer
    to produce a clean, enriched, and business-ready dataset.

**Usage**:
    
    - These views can be queried directly for analytics and reporting.

In [0]:
%sql
--Create Dimension: gold.dim_customers
CREATE OR REPLACE VIEW datalakehouse.gold.dim_customer AS 
SELECT 
    ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key,    -- Surrogate key
    ci.cst_id                           AS customer_id,
    ci.cst_key                          AS customer_number,
    ci.cst_firstname                    AS first_name,
    ci.cst_lastname                     AS last_name,
    la.cntry                            AS country,
    ci.cst_marital_status               AS marital_status,
    CASE
        WHEN ci.cst_gndr != 'n/a' THEN ci.cst_gndr -- CRM is the primary source for gender
        ELSE COALESCE(ca.gen, 'n/a')               -- Fallback to ERP data
    END                                 AS gender,
    ca.bdate                            AS birthdate,
    ci.cst_create_date                  AS create_date
FROM datalakehouse.silver.crm_cust_info ci
LEFT JOIN datalakehouse.silver.erp_cust_az12 ca
    ON ci.cst_key = ca.cid
LEFT JOIN datalakehouse.silver.erp_loc_a101 la
    ON ci.cst_key = la.cid;


--Create Dimension: gold.dim_product
CREATE OR REPLACE VIEW datalakehouse.gold.dim_product AS 
SELECT 
    ROW_NUMBER() OVER (ORDER BY pi.prd_key, pi.prd_start_dt) AS product_key,    -- Surrogate key
    pi.prd_id                           AS product_id,
    pi.prd_key                          AS product_number,
    pi.prd_name                         AS product_name,
    pi.cat_id                           AS category_id,
    ca.cat                              AS category,
    ca.subcat                           AS subcategory,
    ca.maintenance                      AS maintenance,
    pi.prd_cost                         AS cost,
    pi.prd_line                         AS product_line,
    pi.prd_start_dt                     AS start_date
FROM datalakehouse.silver.crm_prd_info pi
LEFT JOIN datalakehouse.silver.erp_px_cat_glv2 ca
    ON pi.cat_id = ca.id
WHERE pi.prd_end_dt IS NULL;


In [0]:
%sql
-- Create Fact Table: gold.fact_sales

CREATE OR REPLACE VIEW datalakehouse.gold.fact_sales AS
SELECT
    sd.sls_ord_num  AS order_number,
    pr.product_key  AS product_key,
    cu.customer_key AS customer_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt  AS shipping_date,
    sd.sls_due_dt   AS due_date,
    sd.sls_sales    AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price    AS price
FROM datalakehouse.silver.crm_sales_details sd
LEFT JOIN datalakehouse.gold.dim_product pr
    ON sd.sls_prd_key = pr.product_number
LEFT JOIN datalakehouse.gold.dim_customer cu
    ON sd.sls_cust_id = cu.customer_id;